# 实验 3.3 昇腾 CANN 基础操作实验

> **实验定位**：本实验从"自下而上、层层依赖"的协同架构视角，系统讲解**昇腾 CANN、操作系统与驱动程序**三者之间的关系，并深入剖析 CANN 软件栈的核心模块与功能层次。在云沙箱环境中，学生将通过运行官方示例与查询命令，完成 NPU 硬件识别、CANN 环境验证、ACL 编程体验等基础操作；最后通过在昇腾香橙派开发板上查询 NPU 与 CANN 版本信息，建立"固件→驱动→CANN"严格匹配的工程意识。

**运行环境**：CANN 9.0.0 · Python 3.11 · Atlas A2 · **Ascend 910B3**（1*NPU）· 16 vCPUs · 32 GiB

> 本案例在 Notebook 下运行，所有代码单元均可直接执行（**Shift+Enter**），建议从头依次运行。本实验侧重 **"关系理解 + 模块认知 + 基础操作 + 香橙派查询"**——通过讲解与动手相结合，建立对昇腾软件栈从硬件到应用的完整认知。

![昇腾全栈](../../images/ascend_all.png)

---

## 学习目标

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">目标</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">理解三者关系</td>
<td style="text-align: left;">弄清昇腾 CANN、操作系统与驱动程序之间自下而上的协同架构</td>
</tr>
<tr>
<td style="text-align: left;">掌握核心模块</td>
<td style="text-align: left;">了解 CANN 四层软件栈及其核心模块（AscendCL、GE、ATC、AOL 等）的功能</td>
</tr>
<tr>
<td style="text-align: left;">环境快速确认</td>
<td style="text-align: left;">在云沙箱中确认 Python、PyTorch、torch_npu 与 NPU 硬件就绪</td>
</tr>
<tr>
<td style="text-align: left;">硬件信息查询</td>
<td style="text-align: left;">使用 npu-smi 查询 NPU 芯片型号、健康状态、温度功耗等</td>
</tr>
<tr>
<td style="text-align: left;">CANN 环境验证</td>
<td style="text-align: left;">检查 CANN 版本、环境变量、目录结构，验证软件栈配置正确</td>
</tr>
<tr>
<td style="text-align: left;">ACL 编程体验</td>
<td style="text-align: left;">通过 HelloWorld 示例理解 ACL 编程基本流程，验证端到端通路</td>
</tr>
<tr>
<td style="text-align: left;">香橙派查询验证</td>
<td style="text-align: left;">在昇腾香橙派上查询 NPU 与 CANN 版本信息</td>
</tr>
</table>

## 学习路径

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">步骤</th>
<th style="text-align: left;">内容</th>
<th style="text-align: left;">学习目标</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">三者关系讲解</td>
<td style="text-align: left;">理解 CANN、操作系统与驱动程序的协同架构</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">CANN 核心模块</td>
<td style="text-align: left;">了解 CANN 四层软件栈及各层核心模块功能</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">环境快速确认</td>
<td style="text-align: left;">确认基础环境就绪，为后续操作做准备</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">CANN 软件栈探查</td>
<td style="text-align: left;">探查 CANN 安装目录结构与关键环境变量</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">NPU 硬件信息查询</td>
<td style="text-align: left;">使用 npu-smi 查询 NPU 硬件状态</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">CANN 运行环境验证</td>
<td style="text-align: left;">检查版本、环境变量、目录结构</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">ACL 编程与 HelloWorld</td>
<td style="text-align: left;">验证 CANN 软件栈端到端通路</td>
</tr>
<tr>
<td style="text-align: left;">8</td>
<td style="text-align: left;">香橙派查询验证</td>
<td style="text-align: left;">查询 NPU 与 CANN 版本信息（见 <code>新建 Microsoft Word 文档.docx</code>）</td>
</tr>
<tr>
<td style="text-align: left;">9</td>
<td style="text-align: left;">总结与思考</td>
<td style="text-align: left;">回顾学习要点，思考后续开发方向</td>
</tr>
</table>

---

## 第1步：昇腾 CANN、操作系统与驱动程序之间的关系

> **学习目标**：理解昇腾 CANN、操作系统与驱动程序之间自下而上、层层依赖的协同架构，建立"固件→驱动→CANN"严格匹配的工程意识。

### 1.1 三者关系的核心概括

昇腾 CANN、操作系统与驱动程序之间的关系，可以概括为一种**自下而上、层层依赖的协同架构**：

- **驱动程序**是连接操作系统与昇腾 NPU 硬件的"桥梁"，它让操作系统能识别并管理 NPU；
- **CANN** 则是构建在操作系统与驱动之上的"AI 计算平台"，它通过调用驱动提供的底层接口来管理和调度 NPU 资源，为上层 AI 应用提供统一编程接口和工具链。

三者紧密联动，其**安装顺序、版本号必须严格匹配**，部署时必须遵循"**固件→驱动→CANN**"的顺序：

```
固件 (Firmware)  →  驱动 (Driver)  →  CANN
  ↑                    ↑                  ↑
  烧录到 NPU            安装到 OS           安装到 OS
  最底层硬件代码         让 OS 识别 NPU      提供 AI 计算平台
```

先确保 NPU 硬件被操作系统正确识别，再搭建 CANN 软件环境，最终由 CANN 向上层（如 PyTorch 等 AI 框架）提供高效的 AI 计算能力。

![异构计算架构](../../images/heterogeneous_computing_architecture.png)

### 1.2 自下而上的四层协同架构

从硬件到应用，整个昇腾计算平台可以分为四层，每一层都依赖其下层提供的服务：

```
┌─────────────────────────────────────────────────────┐
│  第4层  应用层：PyTorch / MindSpore / TensorFlow      │  ← 你写的 AI 模型代码
├─────────────────────────────────────────────────────┤
│  第3层  CANN 平台：AscendCL + GE + ATC + AOL + ...   │  ← AI 计算平台（统一接口/工具链）
├─────────────────────────────────────────────────────┤
│  第2层  操作系统 + 驱动程序：Linux + NPU Driver       │  ← OS 识别并管理 NPU
├─────────────────────────────────────────────────────┤
│  第1层  硬件层：Ascend NPU + 固件 (Firmware)          │  ← 实际算力芯片
└─────────────────────────────────────────────────────┘
```

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">层次</th>
<th style="text-align: left;">组件</th>
<th style="text-align: left;">职责</th>
<th style="text-align: left;">依赖关系</th>
</tr>
<tr>
<td style="text-align: left;">第1层</td>
<td style="text-align: left;">NPU 硬件 + 固件</td>
<td style="text-align: left;">提供实际 AI 算力（AI Core、HBM 等）</td>
<td style="text-align: left;">最底层，无依赖</td>
</tr>
<tr>
<td style="text-align: left;">第2层</td>
<td style="text-align: left;">操作系统 + 驱动</td>
<td style="text-align: left;">让 OS 识别 NPU，提供设备管理接口</td>
<td style="text-align: left;">依赖第1层硬件</td>
</tr>
<tr>
<td style="text-align: left;">第3层</td>
<td style="text-align: left;">CANN 平台</td>
<td style="text-align: left;">提供统一编程接口与工具链，调度 NPU 资源</td>
<td style="text-align: left;">依赖第2层驱动接口</td>
</tr>
<tr>
<td style="text-align: left;">第4层</td>
<td style="text-align: left;">AI 框架</td>
<td style="text-align: left;">开发者编写模型代码，调用框架 API</td>
<td style="text-align: left;">依赖第3层 CANN 适配</td>
</tr>
</table>

上表清晰展示了四层架构的职责与依赖关系。**第1层**是 NPU 硬件与固件，提供实际算力；**第2层**是操作系统与驱动程序，让 OS 能识别并管理 NPU；**第3层**是 CANN 平台，通过调用驱动接口调度 NPU 资源，向上提供统一编程接口；**第4层**是 PyTorch 等 AI 框架，开发者在此层编写模型代码。每一层都必须依赖其下层才能正常工作，因此部署时必须严格遵循"固件→驱动→CANN→框架"的自下而上顺序。

### 1.3 版本严格匹配的工程要求

CANN 与底层驱动、固件版本**强绑定**，三者版本号必须严格匹配：

```
CANN 9.0.0  ←→  Driver 23.0.x  ←→  Firmware 1.8.x  ←→  Ascend 910B3 硬件
```

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">组件</th>
<th style="text-align: left;">版本示例</th>
<th style="text-align: left;">安装位置</th>
<th style="text-align: left;">可否用户修改</th>
</tr>
<tr>
<td style="text-align: left;">固件 (Firmware)</td>
<td style="text-align: left;">1.8.x</td>
<td style="text-align: left;">烧录到 NPU 芯片</td>
<td style="text-align: left;">否（出厂固化）</td>
</tr>
<tr>
<td style="text-align: left;">驱动 (Driver)</td>
<td style="text-align: left;">23.0.x</td>
<td style="text-align: left;">操作系统内核模块</td>
<td style="text-align: left;">是（需 root 权限）</td>
</tr>
<tr>
<td style="text-align: left;">CANN Toolkit</td>
<td style="text-align: left;">9.0.0</td>
<td style="text-align: left;">/usr/local/Ascend</td>
<td style="text-align: left;">是</td>
</tr>
<tr>
<td style="text-align: left;">AI 框架适配</td>
<td style="text-align: left;">torch_npu</td>
<td style="text-align: left;">Python 环境</td>
<td style="text-align: left;">是</td>
</tr>
</table>

> ⚠️ **关键工程意识**：
> - **云沙箱环境不允许用户自行修改 CANN 版本**，因为 CANN 与底层驱动、固件强绑定，用户侧修改会破坏环境一致性导致系统异常。
> - 在真实端侧项目（如香橙派）中，升级 CANN 时必须**同步确认驱动与固件版本兼容**，否则会导致 NPU 无法识别或运行时崩溃。
> - 部署顺序错误（如先装 CANN 再装驱动）会导致 CANN 找不到 NPU 设备，安装失败。

![CANN软件架构](../../images/cann_software_architecture.png)

---

## 第2步：CANN 核心模块与功能层次

> **学习目标**：了解 CANN 四层软件栈及其核心模块的功能，建立从应用到芯片的完整软件栈认知。

CANN（Compute Architecture for Neural Networks，神经网络计算架构）是华为为昇腾系列 AI 处理器打造的异构计算架构，通过核心模块的紧密协作，构建了一个**从应用到芯片的完整软件栈**，为 AI 开发者提供高效、便捷的开发平台。

CANN 的核心模块按功能可以分为四大层次：

![CANN软件架构](../../images/cann_software_architecture.png)

### 2.1 👑 应用开发与编程接口层

这是开发者最常接触的顶层模块，主要负责提供统一的编程框架和接口。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模块</th>
<th style="text-align: left;">全称</th>
<th style="text-align: left;">功能描述</th>
</tr>
<tr>
<td style="text-align: left;"><strong>AscendCL</strong></td>
<td style="text-align: left;">Ascend Computing Language（昇腾计算语言）</td>
<td style="text-align: left;">作为统一的编程接口，提供设备管理、内存管理、模型加载与执行、算子加载与执行等 API，供用户开发 AI 应用</td>
</tr>
<tr>
<td style="text-align: left;"><strong>GE</strong></td>
<td style="text-align: left;">Graph Engine（图引擎）</td>
<td style="text-align: left;">计算图编译和运行的控制中心，负责图优化、图编译管理和图执行控制，支持多种 AI 框架计算图到 Ascend 图的转换</td>
</tr>
<tr>
<td style="text-align: left;"><strong>ATC</strong></td>
<td style="text-align: left;">Ascend Tensor Compiler（ATC 编译器）</td>
<td style="text-align: left;">核心模型转换工具，负责将 ONNX 等格式的模型编译优化为昇腾硬件可执行的 .om 离线模型</td>
</tr>
</table>

> 💡 **开发者视角**：AscendCL 是最常用的编程接口（类似 CUDA 的 Runtime API）；ATC 是模型部署阶段最常用的工具（将训练好的模型转为昇腾专用的 .om 格式）；GE 在框架适配层自动工作，开发者通常不直接操作。

```
开发者代码 → [AscendCL API] → [GE 图引擎] → [ATC 编译] → .om 离线模型 → NPU 执行
```

#### 📌 AscendCL 具体操作案例

AscendCL 提供设备管理、内存管理、模型管理、算子管理四大类 API。一个典型的 AscendCL 推理应用完整流程如下：

```cpp
#include "acl/acl.h"

// ① 初始化
aclInit(nullptr);
aclrtSetDevice(0);

// ② 加载 .om 离线模型
int32_t modelId;
aclmdlLoadFromFile("resnet18.om", &modelId);

// ③ 申请 Device 内存并拷贝输入数据
void *devInput;
aclrtMalloc(&devInput, inputSize, ACL_MEM_MALLOC_HUGE_FIRST);
aclrtMemcpy(devInput, inputSize, hostInput, inputSize, ACL_MEMCPY_HOST_TO_DEVICE);

// ④ 执行推理
aclmdlExecute(modelId, inputDataset, outputDataset);

// ⑤ 拷贝输出结果回 Host 并释放资源
aclrtMemcpy(hostOutput, outputSize, devOutput, outputSize, ACL_MEMCPY_DEVICE_TO_HOST);
aclrtFree(devInput);
aclmdlUnload(modelId);
aclrtResetDevice(0);
aclFinalize();
```

> **说明**：本实验第7步的 HelloWorld 示例就是使用 AscendCL API（`aclInit`、`aclrtSetDevice`、`aclrtCreateStream` 等）验证端到端通路的最简案例。实际 AI 推理应用在此基础上增加模型加载与执行步骤。

#### 📌 GE 图引擎具体操作案例

GE 作为图编译和运行的控制中心，通常由框架适配层（如 PyTorch 的 `torch_npu`、MindSpore 昇腾后端）自动调用，开发者一般不直接编写 GE 代码。其工作流程为：

```
PyTorch/MindSpore 计算图  →  GE 接收  →  图优化(算子融合/常量折叠)  →  图编译  →  Ascend 图  →  NPU 执行
```

若需直接使用 GE C++ API 构建计算图，典型代码如下：

```cpp
#include "ge_api.h"
using namespace ge;

// 构建计算图：y = x * w + b
auto x = op::Data("x").set_attr({1, 224, 224, 3});
auto w = op::Const("w").set_value(weight_tensor);
auto b = op::Const("b").set_value(bias_tensor);
auto mul = op::Mul("mul").set_input_x1(x).set_input_x2(w);
auto add = op::Add("add").set_input_x1(mul).set_input_x2(b);

// 编译并运行
ge::Graph graph("my_graph");
graph.SetInputs({x}).SetOutputs({add});
ge::SessionOptions options;
auto session = ge::Session(options);
session.AddGraph(graph_id, graph);
session.RunGraph(graph_id, inputs, outputs);
```

> **说明**：在 PyTorch + torch_npu 环境中，当你执行 `model.npu()(input)` 时，框架自动将 PyTorch 计算图通过 GE 转换为 Ascend 图并在 NPU 上执行，开发者无需感知 GE 的存在。

#### 📌 ATC 编译器具体操作案例

ATC 是命令行工具，最常用的场景是将 ONNX 模型转换为昇腾 .om 离线模型：

```bash
# 将 ONNX 模型转换为 .om 离线模型
atc --model=resnet18.onnx \
    --framework=5 \
    --output=resnet18 \
    --soc_version=Ascend910B3 \
    --input_shape="input:1,3,224,224"

# 参数说明：
#   --model       输入模型路径
#   --framework   框架类型 (5=ONNX, 3=TensorFlow, 1=Caffe)
#   --output      输出 .om 文件名（自动加 .om 后缀）
#   --soc_version 目标芯片型号（必须与实际 NPU 一致）
#   --input_shape 输入张量形状
```

也可从 PyTorch 模型直接转换（需先导出 ONNX）：

```python
# 步骤1：PyTorch 模型导出 ONNX
import torch, torchvision
model = torchvision.models.resnet18(weights='IMAGENET1K_V1').eval()
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, dummy, 'resnet18.onnx', opset_version=11)

# 步骤2：用 ATC 命令行将 ONNX 转为 .om（在终端执行）
# atc --model=resnet18.onnx --framework=5 --output=resnet18 --soc_version=Ascend910B3
```

> **说明**：ATC 转换后的 .om 模型包含了针对特定芯片优化的算子二进制代码，加载速度更快、推理性能更高，是生产部署的推荐格式。`--soc_version` 必须与目标 NPU 型号严格匹配，否则转换失败。

### 2.2 🏗️ 计算服务与编译优化层

这一层负责提供高性能的算子库和编译优化能力。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模块</th>
<th style="text-align: left;">全称</th>
<th style="text-align: left;">功能描述</th>
</tr>
<tr>
<td style="text-align: left;"><strong>AOL</strong></td>
<td style="text-align: left;">Ascend Operator Library（算子加速库）</td>
<td style="text-align: left;">提供丰富的、深度优化的高性能算子，包括神经网络（NN）库、线性代数计算库（BLAS）等</td>
</tr>
<tr>
<td style="text-align: left;"><strong>AOE</strong></td>
<td style="text-align: left;">Ascend Optimization Engine（调优引擎）</td>
<td style="text-align: left;">通过算子、子图、梯度等多维度调优，结合 AMCT 模型压缩工具，提升模型端到端的运行速度</td>
</tr>
<tr>
<td style="text-align: left;"><strong>TBE</strong></td>
<td style="text-align: left;">Tensor Boost Engine</td>
<td style="text-align: left;">自定义算子开发框架，提供自动调度机制，帮助开发者高效地编译和优化自定义算子</td>
</tr>
<tr>
<td style="text-align: left;"><strong>BiSheng Compiler</strong></td>
<td style="text-align: left;">毕昇编译器</td>
<td style="text-align: left;">提供 Host-Device 异构编程编译能力，通过精准编译优化释放昇腾 AI 处理器性能</td>
</tr>
</table>

> 💡 **性能视角**：AOL 提供的预优化算子是开箱即用的高性能保障；当内置算子不��足需求时，可通过 TBE 开发自定义算子；AOE 和 BiSheng Compiler 则在编译期进行深度优化，进一步提升性能。

![AI Core架构](../../images/ai_core_architecture.png)

#### 📌 AOL 算子加速库具体操作案例

AOL 包含两个核心子库：**NN 库**（神经网络算子，如 Conv2d、MatMul、ReLU、Softmax 等）和 **BLAS 库**（线性代数算子，如 GEMM、GEMV 等）。开发者通过 AscendCL 的算子执行 API 调用：

```cpp
#include "acl/acl.h"
#include "acl/ops/acl_cnn.h"   // NN 算子头文件
#include "acl/ops/acl_blas.h"  // BLAS 算子头文件

// 调用 NN 库的 Conv2d 算子
aclStatus ret = aclnnConv2d(workspaceId, xDesc, x, wDesc, w, yDesc, y,
                            strideH, strideW, dilationH, dilationW,
                            padH, padW, group, stream);

// 调用 BLAS 库的 GEMM 算子：C = alpha * A * B + beta * C
ret = aclblasGemm(workspaceId, ACLblasTransNo, ACLblasTransNo,
                  ACLblasTransNo, m, n, k, alpha,
                  aDesc, a, bDesc, b, beta, cDesc, c, stream);
```

> **说明**：在 PyTorch + torch_npu 环境中，当你调用 `torch.nn.functional.conv2d()` 或 `torch.mm()` 时，torch_npu 会自动将请求映射到 AOL 中对应的高性能算子实现，开发者无需直接调用 AOL API。AOL 中的算子均由华为工程师针对昇腾 AI Core 架构深度优化，性能远超通用实现。

#### 📌 AOE 调优引擎具体操作案例

AOE 提供算子级、子图级、整网级三个维度的自动调优，通常通过命令行工具调用：

```bash
# 算子级调优：对单个算子进行自动调优
aoe --mode=op --framework=5 --model=matmul.onnx \
    --soc_version=Ascend910B3 --output=matmul_tuned

# 子图级调优：对计算子图进行融合优化
aoe --mode=subgraph --framework=5 --model=subgraph.onnx \
    --soc_version=Ascend910B3 --output=subgraph_tuned

# 模型压缩（AMCT 工具）：将 FP16 模型量化为 INT8
amct_onnx calibration --model=resnet18.onnx \
    --save_path=resnet18_int8 --data_dir=calibration_data/
```

> **说明**：AOE 调优过程会自动搜索最优的算子切分策略、数据排布和并行度，生成调优后的模型。AMCT 模型压缩工具可将 FP16/FP32 模型量化为 INT8，在精度损失可控的前提下大幅提升推理速度、降低显存占用。

#### 📌 TBE 自定义算子开发案例

当 AOL 内置算子不满足需求时（如新型激活函数、特殊注意力机制），可通过 TBE 框架开发自定义算子：

```python
from tbe import tbe
from tbe.lang import register_op

# 定义自定义 Swish 激活函数算子：y = x * sigmoid(x)
@register_op("swish", "util.swish", "Swish")
def swish_compute(x, beta=1.0):
    """Swish 激活函数计算逻辑"""
    sigmoid_x = tbe.lang.vsigmoid(tbe.lang.vmuls(x, beta))  # sigmoid(beta*x)
    result = tbe.lang.vmul(x, sigmoid_x)                     # x * sigmoid(beta*x)
    return result

# 算子信息注册（输入输出描述、数据类型支持等）
def swish_info():
    op_info = tbe.lang.OpInfo("swish") \
        .input(0, "x", "required", "all") \
        .output(0, "y", "required", "all") \
        .attr("beta", "optional", "float", "1.0")
    return op_info
```

> **说明**：TBE 提供自动调度机制，开发者只需描述算子的计算语义（用 `vmul`、`vsigmoid` 等 Tensor API），TBE 编译器自动生成针对 AI Core 的最优调度代码。开发完成后，通过 `op_build` 编译为 .o 二进制，再注册到 CANN 算子库中即可在模型中使用。

#### 📌 BiSheng Compiler 毕昇编译器具体操作案例

BiSheng Compiler 支持 Host-Device 异构编程，开发者可在同一源文件中编写 Host 侧控制逻辑和 Device 侧计算核函数：

```cpp
// bisheng_swish.bsh —— 毕昇异构源码
#include <bisheng/bisheng.hpp>
using namespace bisheng;

// Device 核函数：在 AI Core 上执行
KERNEL void swish_kernel(float* x, float* y, int n, float beta) {
    int i = GetGlobalId(0);
    if (i < n) {
        float val = x[i];
        y[i] = val / (1.0f + expf(-beta * val));  // Swish 激活
    }
}

// Host 函数：在 CPU 上执行，负责下发任务到 NPU
extern "C" int swish_op(float* x, float* y, int n, float beta) {
    // 自动生成内存搬运和任务调度代码
    LaunchKernel(swish_kernel, {n}, x, y, n, beta);
    return 0;
}
```

```bash
# 编译毕昇源码为可执行算子
bisheng bisheng_swish.bsh -o swish_op.o --soc=Ascend910B3
```

> **说明**：BiSheng Compiler 的优势在于 Host-Device 统一编程模型——开发者无需分别编写 Host 和 Device 代码，编译器自动处理内存搬运、任务调度和同步。相比 TBE，BiSheng 更接近 C++ 原生语法，学习成本更低，同时通过精准编译优化释放硬件性能。

### 2.3 ⚙️ 运行时与执行层

这一层负责管理硬件资源和执行具体任务。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模块</th>
<th style="text-align: left;">全称</th>
<th style="text-align: left;">功能描述</th>
</tr>
<tr>
<td style="text-align: left;"><strong>Runtime</strong></td>
<td style="text-align: left;">运行时</td>
<td style="text-align: left;">提供高效的硬件资源管理，包括模型推理、单算子加载执行等开发接口，负责将任务实际调度到 NPU 上执行</td>
</tr>
<tr>
<td style="text-align: left;"><strong>HCCL</strong></td>
<td style="text-align: left;">Huawei Collective Communication Library（集合通信库）</td>
<td style="text-align: left;">基于昇腾硬件的高性能集合通信库，支持单机多卡、多机多卡间的数据并行和模型并行</td>
</tr>
<tr>
<td style="text-align: left;"><strong>DVPP</strong></td>
<td style="text-align: left;">Digital Vision Pre-Processing（数字视觉预处理）</td>
<td style="text-align: left;">负责图像/视频编解码、缩放等硬件加速的视觉预处理</td>
</tr>
<tr>
<td style="text-align: left;"><strong>AIPP</strong></td>
<td style="text-align: left;">AI Pre-Processing（AI 预处理）</td>
<td style="text-align: left;">负责 AI 推理前的归一化、通道交换等硬件加速预处理</td>
</tr>
</table>

> 💡 **执行视角**：Runtime 是任务调度的核心，将上层下发的计算任务实际分配到 NPU 的 AI Core 上执行；HCCL 在分布式训练中至关重要；DVPP 与 AIPP 将数据预处理从 CPU 卸载到 NPU，减少数据搬运开销。

![Host与Device](../../images/host_and_device.png)

#### 📌 Runtime 运行时具体操作案例

Runtime 提供设备管理、Stream/Event 管理、内存管理、模型推理等接口。AscendCL 的运行时 API（以 `aclrt` 为前缀）即封装了 Runtime 能力：

```cpp
#include "acl/acl.h"

// ① 设备管理
int32_t deviceId = 0;
aclrtSetDevice(deviceId);           // 指定当前线程使用的 NPU 设备
aclrtGetCurrentDevice(&deviceId);   // 查询当前设备 ID

// ② Stream 管理（异步执行流）
aclrtStream stream;
aclrtCreateStream(&stream);         // 创建 Stream
aclrtSynchronizeStream(stream);     // 等待 Stream 上所有任务完成
aclrtDestroyStream(stream);         // 销毁 Stream

// ③ Event 管理（用于 Stream 间同步）
aclrtEvent event;
aclrtCreateEvent(&event);
aclrtRecordEvent(event, stream1);   // 在 stream1 上记录 Event
aclrtWaitEvent(event, stream2);     // stream2 等待 Event 完成

// ④ 内存管理（Host-Device 数据搬运）
void *devPtr;
aclrtMalloc(&devPtr, size, ACL_MEM_MALLOC_HUGE_FIRST);  // 申请 Device 内存
aclrtMemcpy(devPtr, size, hostPtr, size, ACL_MEMCPY_HOST_TO_DEVICE);  // H2D 拷贝
aclrtFree(devPtr);                 // 释放 Device 内存
```

> **说明**：在 PyTorch + torch_npu 中，`torch.npu.synchronize()` 对应 `aclrtSynchronizeStream()`，`tensor.npu()` 触发 `aclrtMemcpy` 的 H2D 拷贝。Runtime 是 AscendCL 的底层支撑，所有 AscendCL API 最终都通过 Runtime 调度到 NPU。

#### 📌 HCCL 集合通信库具体操作案例

HCCL 用于单机多卡/多机多卡的分布式训练，提供 AllReduce、AllGather、Broadcast 等集合通信原语，类似 NVIDIA 的 NCCL：

```cpp
#include "hccl/hccl.h"

// ① 初始化 HCCL
HcclComm comm;
HcclCommInitRootInfo(rootId, &comm);  // 以 rank 0 为 root 初始化通信域

// ② AllReduce：多卡梯度求和（分布式训练最常用）
void *sendbuf, *recvbuf;  // Device 内存中的梯度数据
HcclAllReduce(sendbuf, recvbuf, count, HCCL_DATA_TYPE_FP32,
              HCCL_REDUCE_SUM, comm, stream);

// ③ AllGather：多卡特征拼接
HcclAllGather(sendbuf, recvbuf, sendCount, HCCL_DATA_TYPE_FP16, comm, stream);

// ④ Broadcast：参数广播（将 rank 0 的模型参数广播到所有卡）
HcclBroadcast(buff, count, HCCL_DATA_TYPE_FP32, rootRank, comm, stream);
```

> **说明**：在 PyTorch 分布式训练中，`torch.distributed.init_process_group(backend='hccl')` 即使用 HCCL 后端。HCCL 充分利用昇腾硬件的 RoCE/VPC 高速互联链路，通信效率远超基于 TCP 的 Gloo 后端。

#### 📌 DVPP 数字视觉预处理具体操作案例

DVPP 将图像/视频的编解码、缩放、裁剪等操作从 CPU 卸载到 NPU 的专用硬件模块执行，减少数据搬运开销：

```cpp
#include "acl/ops/acl_dvpp.h"

// ① 创建 DVPP 通道
aclvdecChannelDesc vdecChannelDesc = aclvdecCreateChannelDesc();
aclvdecSetChannelDescChannelId(vdecChannelDesc, 10);

// ② 图像解码：JPEG → YUV420
aclvdecSendFrame(vdecChannelDesc, picDesc, stream);

// ③ 图像缩放：1920x1080 → 224x224（硬件加速）
acldvppVpcResizeAsync(dvppChannelDesc, inputDesc, outputDesc,
                      resizeConfig, stream);

// ④ 图像裁剪
acldvppVpcCropAsync(dvppChannelDesc, inputDesc, outputDesc,
                     cropArea, stream);
```

> **说明**：DVPP 适合视频流推理场景（如智能摄像头、安防监控），在 NPU 上直接解码 H.264/H.265 视频帧并缩放到模型输入尺寸，避免视频帧在 CPU 与 NPU 之间来回搬运。

#### 📌 AIPP AI 预处理具体操作案例

AIPP 在模型推理前执行归一化、通道交换（RGB↔BGR）、色域转换、减均值等操作，直接融合在 .om 模型中，无需额外编写预处理代码：

```protobuf
# aipp.cfg —— AIPP 配置文件
aipp_op {
    aipp_mode : static
    input_format : YUV420SP_U8       # 输入格式：DVPP 输出的 YUV420
    src_image_size_w : 224           # 输入图像宽
    src_image_size_h : 224           # 输入图像高
    csc_switch : true                # 色域转换：YUV → RGB
    rbuv_swap_switch : false         # R/B 通道交换
    matrix_r0c0 : 1.0                # CSC 矩阵系数
    mean_chn_0 : 123.675             # 减均值（ImageNet 均值）
    mean_chn_1 : 116.28
    mean_chn_2 : 103.53
    var_reci_chn_0 : 0.01712475      # 乘方差倒数（ImageNet 标准差）
    var_reci_chn_1 : 0.01750700
    var_reci_chn_2 : 0.01442940
}
```

```bash
# 在 ATC 转换时插入 AIPP 预处理
atc --model=resnet18.onnx --framework=5 --output=resnet18_aipp \
    --soc_version=Ascend910B3 --insert_op_conf=aipp.cfg
```

> **说明**：AIPP 配置后，模型直接接收 DVPP 输出的 YUV420 图像，AIPP 在 NPU 上自动完成 YUV→RGB→减均值→除标准差的全流程，无需在 CPU 上做预处理，实现"原始图像进 NPU，推理结果出 NPU"的零 CPU 预处理流水线。

### 2.4 🧱 基础与硬件抽象层

这是最底层，负责屏蔽硬件差异，提供基础服务。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模块</th>
<th style="text-align: left;">功能描述</th>
</tr>
<tr>
<td style="text-align: left;"><strong>HAL</strong>（硬件抽象层）</td>
<td style="text-align: left;">屏蔽底层硬件差异，提供统一的驱动接口</td>
</tr>
<tr>
<td style="text-align: left;"><strong>SVM</strong>（共享虚拟内存）</td>
<td style="text-align: left;">实现主机与设备间共享虚拟内存，简化数据搬运</td>
</tr>
<tr>
<td style="text-align: left;"><strong>VM</strong>（设备虚拟化）</td>
<td style="text-align: left;">提供设备虚拟化能力，支持多租户共享 NPU</td>
</tr>
<tr>
<td style="text-align: left;"><strong>HDC</strong>（主机-设备通信）</td>
<td style="text-align: left;">提供主机与设备之间的高效通信通道</td>
</tr>
</table>

> 💡 **架构视角**：HAL 是 CANN 跨硬件兼容的关键——不同型号的昇腾芯片（如 310B4、910B3）通过 HAL 向上层呈现统一接口，使得同一套 AscendCL 代码可以在不同芯片上运行。SVM、VM、HDC 等基础服务为上层模块提供底层支撑。


#### 📌 HAL 硬件抽象层具体操作案例

HAL 对开发者完全透明，无需直接调用。它的工作体现在：当你在不同型号 NPU 上运行同一份代码时，HAL 内部自动适配硬件差异。

```
开发者调用 aclrtSetDevice(0)
        ↓
AscendCL 将请求转发给 HAL
        ↓
HAL 根据当前芯片型号选择对应的驱动接口：
  ├── Ascend 910B3 → 调用 910B3 驱动的 set_device()
  ├── Ascend 310B4 → 调用 310B4 驱动的 set_device()
  └── Ascend 910P3 → 调用 910P3 驱动的 set_device()
        ↓
驱动操作实际硬件
```

> **说明**：本实验中云沙箱使用 910B3，香橙派使用 310B4，两者的 `npu-smi info` 命令、ACL API、ATC 工具用法完全一致，这正是 HAL 的功劳。开发者只需在 ATC 转换时指定 `--soc_version`，HAL 负责其余所有硬件适配。HAL 的存在使得昇腾软件栈实现了"一次开发，多芯片运行"的跨平台能力。

#### 📌 SVM 共享虚拟内存具体操作案例

SVM（Shared Virtual Memory）允许 Host 和 Device 共享同一块虚拟内存地址空间，简化数据搬运。在传统模式下，开发者需手动 `aclrtMalloc` + `aclrtMemcpy` 在 Host 和 Device 间拷贝数据；SVM 模式下，Host 和 Device 直接访问同一地址：

```cpp
#include "acl/acl.h"

// 传统模式：需手动拷贝
void *hostPtr = malloc(size);
void *devPtr;
aclrtMalloc(&devPtr, size, ACL_MEM_MALLOC_HUGE_FIRST);
aclrtMemcpy(devPtr, size, hostPtr, size, ACL_MEMCPY_HOST_TO_DEVICE);  // 显式拷贝
// ... Device 计算 ...
aclrtMemcpy(hostPtr, size, devPtr, size, ACL_MEMCPY_DEVICE_TO_HOST);  // 显式拷贝回

// SVM 模式：Host 和 Device 共享同一地址
void *svmPtr;
aclrtMallocSVM(&svmPtr, size, ACL_MEM_MALLOC_HUGE_FIRST);  // 申请 SVM 内存
// Host 侧直接写入
memset(svmPtr, 0, size);
// Device 侧直接读取（无需拷贝）
aclrtSynchronizeStream(stream);  // 确保 Host 写入完成后 Device 再读
// Device 计算结果 Host 侧直接可见
aclrtFreeSVM(svmPtr);
```

> **说明**：SVM 模式下 Host 和 Device 使用统一的虚拟地址，省去了显式的 `aclrtMemcpy` 拷贝调用，简化编程模型。在 PyTorch + torch_npu 中，部分零拷贝优化即利用了 SVM 能力。SVM 特别适合频繁的小数据交互场景。

#### 📌 VM 设备虚拟化具体操作案例

VM（Virtualization Management）提供 NPU 设备虚拟化能力，支持将一颗物理 NPU 切分为多个虚拟 NPU（vNPU），分配给不同租户或容器使用：

```bash
# 查看物理 NPU 信息
npu-smi info -t board

# 创建 vNPU（将物理 NPU 0 切分为 2 个 vNPU，各占 50% 算力和 16GB 显存）
npu-smi set -i 0 -t vf-create -f 2 -c 50 -m 16384

# 查看已创建的 vNPU
npu-smi info -t vf

# 在容器中绑定 vNPU
docker run --device /dev/davinci0_container1 ...
docker run --device /dev/davinci0_container2 ...
```

> **说明**：VM 是云服务场景的关键能力——云沙箱环境正是通过 VM 将物理 NPU 虚拟化后分配给各用户实例。在端侧场景中，VM 支持多个 AI 应用共享同一颗 NPU，实现资源隔离与按需分配。开发者通过 `aclrtSetDevice(vNpuId)` 选择使用哪个 vNPU。

#### 📌 HDC 主机-设备通信具体操作案例

HDC（Host-Device Communication）提供 Host CPU 与 NPU 之间的高效通信通道，负责下发计算任务、传递控制消息和同步事件。HDC 对开发者透明，由 Runtime 内部调用：

```
Host CPU                          NPU (Device)
  │                                  │
  │  ① aclrtSetDevice(0)             │
  │ ───HDC 控制通道──→               │
  │                                  │
  │  ② aclrtMemcpy(H2D)              │
  │ ───HDC 数据通道──→ [Device 内存]  │
  │                                  │
  │  ③ aclmdlExecute()               │
  │ ───HDC 控制通道──→ [AI Core 执行] │
  │                                  │
  │  ④ aclrtSynchronizeStream()      │
  │ ←──HDC 中断回调───  [执行完成]    │
  │                                  │
  │  ⑤ aclrtMemcpy(D2H)              │
  │ ←──HDC 数据通道───  [结果回传]     │
```

> **说明**：HDC 分为**控制通道**（下发任务指令、同步控制）和**数据通道**（传输计算数据）两类。控制通道走 PCIe 控制线，延迟低但带宽小；数据通道走 PCIe 数据线或 HBM 直访，带宽大。在 PyTorch + torch_npu 中，每次 `model(input)` 调用都会触发 HDC 下发计算图到 NPU，`torch.npu.synchronize()` 等待 HDC 回传完成通知。HDC 的通信效率直接影响端到端推理延迟，是 CANN 性能优化的重点之一。

### 2.5 CANN 软件栈全景总结

CANN 正是通过以上这些核心模块的紧密协作，构建了一个从应用到芯片的完整软件栈：

```
┌─────────────────────────────────────────────────────────────┐
│  👑 应用开发与编程接口层：AscendCL · GE · ATC                │
├─────────────────────────────────────────────────────────────┤
│  🏗️ 计算服务与编译优化层：AOL · AOE · TBE · BiSheng Compiler  │
├─────────────────────────────────────────────────────────────┤
│  ⚙️ 运行时与执行层：Runtime · HCCL · DVPP · AIPP             │
├─────────────────────────────────────────────────────────────┤
│  🧱 基础与硬件抽象层：HAL · SVM · VM · HDC                   │
├─────────────────────────────────────────────────────────────┤
│  驱动层：NPU Driver + Firmware                               │
├─────────────────────────────────────────────────────────────┤
│  硬件层：Ascend NPU（如 910B3 / 310B4）                      │
└─────────────────────────────────────────────────────────────┘
```

为 AI 开发者提供了高效、便捷的开发平台。

---

### 2.6 贯穿全栈的综合案例：ResNet18 图像分类端到端流水线

> 为了将上述四层模块从"抽象概念"落到"具体代码"，下面以一个**ResNet18 图像分类**的完整端到端流水线为例，展示从模型训练到 NPU 推理部署的全过程，并标注每个 CANN 模块在其中扮演的角色。

#### 案例场景

```
输入：一张 1920×1080 的 JPEG 猫咪照片
目标：在昇腾 NPU 上完成推理，输出 Top-5 分类结果
模型：ResNet18（ImageNet 1000 类）
```

#### 全栈流水线全景

```
┌─────────────────────────────────────────────────────────────────────────┐
│  阶段0  训练 (Host CPU/GPU)                                             │
│    PyTorch 训练 ResNet18 → 导出 resnet18.onnx                          │
├─────────────────────────────────────────────────────────────────────────┤
│  阶段1  模型转换 (Host CPU, ATC 工具)                                   │
│    resnet18.onnx ──[ATC]──→ resnet18.om (含 AIPP 预处理)               │
├─────────────────────────────────────────────────────────────────────────┤
│  阶段2  推理部署 (Host CPU + NPU, AscendCL API)                        │
│    加载 .om → DVPP 解码 JPEG → AIPP 预处理 → NPU 推理 → 输出 Top-5    │
└─────────────────────────────────────────────────────────────────────────┘
```

#### 阶段0：训练并导出 ONNX 模型

```python
import torch, torchvision

# 加载预训练 ResNet18
model = torchvision.models.resnet18(weights='IMAGENET1K_V1').eval()

# 导出 ONNX（后续交给 ATC 转为 .om）
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, dummy, 'resnet18.onnx', opset_version=11)
print('✅ 已导出 resnet18.onnx')
```

#### 阶段1：ATC 模型转换（融入 AIPP 预处理）

```bash
# 编写 AIPP 配置文件 aipp.cfg：将 DVPP 输出的 YUV420 自动转为 RGB 并归一化
cat > aipp.cfg << 'EOF'
aipp_op {
    aipp_mode : static
    input_format : YUV420SP_U8
    src_image_size_w : 224
    src_image_size_h : 224
    csc_switch : true               # YUV → RGB 色域转换
    mean_chn_0 : 123.675            # 减均值
    mean_chn_1 : 116.28
    mean_chn_2 : 103.53
    var_reci_chn_0 : 0.01712475     # 乘标准差倒数
    var_reci_chn_1 : 0.01750700
    var_reci_chn_2 : 0.01442940
}
EOF

# ATC 转换：ONNX → .om，同时插入 AIPP
atc --model=resnet18.onnx     --framework=5     --output=resnet18_aipp     --soc_version=Ascend910B3     --input_shape="input:1,3,224,224"     --insert_op_conf=aipp.cfg

# 结果：生成 resnet18_aipp.om（已融合预处理，直接接收 YUV420 图像）
```

#### 阶段2：AscendCL 推理应用（完整 C 代码）

```cpp
#include "acl/acl.h"
#include "acl/ops/acl_dvpp.h"

int main() {
    // ========== 初始化 ==========
    aclInit(nullptr);                          // AscendCL: 初始化运行时
    aclrtSetDevice(0);                         // Runtime:   指定 NPU 设备
    aclrtStream stream;
    aclrtCreateStream(&stream);                // Runtime:   创建执行流

    // ========== 加载 .om 模型 ==========
    int32_t modelId;
    aclmdlLoadFromFile("resnet18_aipp.om", &modelId);  // AscendCL: 加载离线模型
    //   └─ GE: 模型加载时 GE 解析 .om 中的计算图结构

    // ========== DVPP 硬件解码 JPEG ==========
    aclvdecChannelDesc *vdecChan = aclvdecCreateChannelDesc();
    aclvdecSetChannelDescChannelId(vdecChan, 0);
    // ... 配置 JPEG 解码参数 ...
    aclvdecSendFrame(vdecChan, jpegInput, stream);     // DVPP:  JPEG → YUV420 (NPU 硬件解码)

    // ========== DVPP 硬件缩放 1920×1080 → 224×224 ==========
    acldvppVpcResizeAsync(dvppChan, yuvInput, yuvResized,
                          resizeCfg, stream);          // DVPP:  硬件加速缩放

    // ========== 模型推理（AIPP 已融合在 .om 中） ==========
    aclmdlExecute(modelId, inputDataset, outputDataset);  // AscendCL: 执行推理
    //   ├─ AIPP:  YUV420 → RGB → 减均值 → 除标准差 (NPU 硬件预处理)
    //   ├─ AOL:   Conv2d/MatMul/ReLU 等算子调用高性能库实现
    //   ├─ GE:    调度计算图执行顺序
    //   └─ Runtime: 将算子下发到 AI Core 执行

    // ========== 拷贝结果回 Host ==========
    aclrtMemcpy(hostOutput, outSize, devOutput, outSize,
                ACL_MEMCPY_DEVICE_TO_HOST);     // Runtime: D2H 数据搬运
    //   └─ HDC: 底层 Host-Device 通信通道实际传输数据

    // ========== 解析 Top-5 结果 ==========
    // hostOutput 中是 1000 类的 logits，CPU 上做 softmax + topk
    print_top5(hostOutput);

    // ========== 释放资源 ==========
    aclmdlUnload(modelId);
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return 0;
}
```

#### 各模块在本案例中的角色一览

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">层次</th>
<th style="text-align: left;">模块</th>
<th style="text-align: left;">在本案例中的具体作用</th>
<th style="text-align: left;">触发方式</th>
</tr>
<tr>
<td style="text-align: left;"><strong>👑 接口层</strong></td>
<td style="text-align: left;"><strong>AscendCL</strong></td>
<td style="text-align: left;">提供 <code>aclInit</code>/<code>aclmdlLoadFromFile</code>/<code>aclmdlExecute</code> 等 API，是开发者唯一直接调用的接口</td>
<td style="text-align: left;">显式调用</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>GE</strong></td>
<td style="text-align: left;">加载 .om 时解析计算图结构，推理时调度算子执行顺序</td>
<td style="text-align: left;">自动触发</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>ATC</strong></td>
<td style="text-align: left;">阶段1 将 ONNX 编译为 .om，同时将 AIPP 预处理融合进模型</td>
<td style="text-align: left;">命令行工具</td>
</tr>
<tr>
<td style="text-align: left;"><strong>🏗️ 优化层</strong></td>
<td style="text-align: left;"><strong>AOL</strong></td>
<td style="text-align: left;">推理时 Conv2d、MatMul、ReLU 等算子调用 AOL 中预优化的高性能实现</td>
<td style="text-align: left;">自动调用</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>AOE</strong></td>
<td style="text-align: left;">ATC 转换时进行图级调优（算子融合、常量折叠），生成更优的 .om</td>
<td style="text-align: left;">ATC 内部</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>TBE</strong></td>
<td style="text-align: left;">若 ResNet18 含自定义算子，TBE 负责编译该算子的 AI Core 二进制</td>
<td style="text-align: left;">按需触发</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>BiSheng</strong></td>
<td style="text-align: left;">替代 TBE 编译自定义算子（若使用毕昇编程模型）</td>
<td style="text-align: left;">按需触发</td>
</tr>
<tr>
<td style="text-align: left;"><strong>⚙️ 执行层</strong></td>
<td style="text-align: left;"><strong>Runtime</strong></td>
<td style="text-align: left;">管理 NPU 设备、Stream、内存；执行 <code>aclrtMemcpy</code> 数据搬运</td>
<td style="text-align: left;">AscendCL 内部</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>HCCL</strong></td>
<td style="text-align: left;">本案例为单卡推理不涉及；若多卡分布式推理则用于多卡间通信</td>
<td style="text-align: left;">按需触发</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>DVPP</strong></td>
<td style="text-align: left;">阶段2 中 JPEG 解码 + 图像缩放，全部在 NPU 硬件上执行</td>
<td style="text-align: left;">显式调用</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>AIPP</strong></td>
<td style="text-align: left;">推理前自动完成 YUV→RGB→减均值→除标准差，已融合在 .om 中</td>
<td style="text-align: left;">ATC 配置</td>
</tr>
<tr>
<td style="text-align: left;"><strong>🧱 基础层</strong></td>
<td style="text-align: left;"><strong>HAL</strong></td>
<td style="text-align: left;">屏蔽 910B3/310B4 硬件差异，同一份 .om 可在不同芯片运行</td>
<td style="text-align: left;">完全透明</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>SVM</strong></td>
<td style="text-align: left;">若使用 SVM 内存，Host/Device 共享地址省去显式拷贝</td>
<td style="text-align: left;">可选优化</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>VM</strong></td>
<td style="text-align: left;">云沙箱中 NPU 已被虚拟化分配，本案例运行在 vNPU 上</td>
<td style="text-align: left;">环境配置</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"><strong>HDC</strong></td>
<td style="text-align: left;">每次 <code>aclrtMemcpy</code>/<code>aclmdlExecute</code> 底层由 HDC 完成实际数据/指令传输</td>
<td style="text-align: left;">完全透明</td>
</tr>
</table>

#### 简化版：PyTorch + torch_npu 一键推理

> 上述 AscendCL C 代码是**生产部署**方式。在**开发调试**阶段，可以用 PyTorch + torch_npu 大幅简化代码——CANN 各模块由 torch_npu 自动调用：

```python
import torch, torch_npu, torchvision
from PIL import Image
import torchvision.transforms as T

# 加载模型并搬到 NPU（GE 自动将 PyTorch 图转为 Ascend 图）
model = torchvision.models.resnet18(weights='IMAGENET1K_V1').eval().npu()

# 图像预处理（CPU 上执行，对应 AIPP 的软件版本）
img = Image.open('cat.jpg')
transform = T.Compose([T.Resize(256), T.CenterCrop(224),
                       T.ToTensor(),
                       T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
input_tensor = transform(img).unsqueeze(0).npu()   # .npu() 触发 HDC 数据搬运

# NPU 推理（AOL 算子库执行，Runtime 调度，HAL 适配硬件）
with torch.no_grad():
    output = model(input_tensor)           # AscendCL 推理
    torch.npu.synchronize()                # 等待 NPU 完成

# Top-5 结果
probs = torch.nn.functional.softmax(output[0].cpu(), dim=0)
top5 = torch.topk(probs, 5)
print('Top-5:', top5)
```

```
代码对比：
  生产部署 (AscendCL C)  →  ~80 行 C 代码，极致性能，含 DVPP/AIPP 硬件预处理
  开发调试 (PyTorch)     →  ~10 行 Python，开发便捷，预处理在 CPU 上执行
```

> 💡 **核心认知**：无论用 AscendCL C 还是 PyTorch Python，底层都是同一套 CANN 软件栈在工作——AscendCL/GE/ATC 负责接口与编译，AOL/AOE 负责算子与优化，Runtime/DVPP/AIPP 负责执行与预处理，HAL/SVM/HDC 负责硬件抽象与通信。**选择哪种开发方式取决于场景**：研究阶段用 PyTorch 快速验证，生产部署用 AscendCL + ATC + DVPP/AIPP 追求极致性能。

![昇腾全栈](../../images/ascend.png)

---

## 第3步：环境快速确认

> **学习目标**：快速确认 Python、PyTorch、torch_npu 与 NPU 硬件就绪，为后续运行官方示例做准备。

本实验的基础环境已预先调通，我们只需快速确认关键组件可用即可。这一步对应 CANN 协同架构中"第4层 AI 框架"的就绪性验证。

In [ ]:
import sys, os, time, subprocess
print('=' * 60)
print('【环境快速确认】')
print('=' * 60)
print(f'Python: {sys.version.split()[0]}')

import torch
print(f'PyTorch: {torch.__version__}')

NPU_OK = False
try:
    import torch_npu
    print(f'torch_npu: {torch_npu.__version__}')
    if torch.npu.is_available():
        _t = torch.ones(2, 2).npu()
        torch.npu.synchronize()
        NPU_OK = True
        print(f'NPU 设备: {torch.npu.get_device_name(0)}')
        print('✅ NPU 可用，后续将在 NPU 上运行推理')
    else:
        print('⚠️ NPU 不可用，将使用 CPU 运行推理')
except ImportError:
    print('⚠️ torch_npu 未安装，将使用 CPU 运行推理')

import numpy as np
print(f'NumPy: {np.__version__}')
print(f'\n👉 NPU_OK = {NPU_OK}')
print('环境确认完成，可以开始后续操作。')

---

## 第4步：CANN 软件栈结构探查

> **学习目标**：探查 CANN 安装目录结构与关键环境变量，建立对 CANN 软件栈物理布局的直观认知。

### 4.1 NPU 为什么能加速？

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">对比项</th>
<th style="text-align: left;">CPU</th>
<th style="text-align: left;">NPU (Ascend 910B3)</th>
</tr>
<tr>
<td style="text-align: left;">计算单元</td>
<td style="text-align: left;">少量通用 ALU</td>
<td style="text-align: left;">数十个 AI Core（Vector + Cube）</td>
</tr>
<tr>
<td style="text-align: left;">并行方式</td>
<td style="text-align: left;">串行为主</td>
<td style="text-align: left;">大规模数据并行</td>
</tr>
<tr>
<td style="text-align: left;">精度优化</td>
<td style="text-align: left;">FP64 为主</td>
<td style="text-align: left;">FP16/BF16/INT8 优化</td>
</tr>
<tr>
<td style="text-align: left;">内存带宽</td>
<td style="text-align: left;">DDR（数十 GB/s）</td>
<td style="text-align: left;">HBM（数百 GB/s）</td>
</tr>
</table>

> 💡 **关键认知**：在昇腾平台上，通过 `torch_npu` 只需加一个 `.npu()` 即可将模型和数据搬到 NPU 上运行，开发体验与 CUDA 几乎一致。

![计算单元](../../images/computing_unit.png)

In [ ]:
print('=' * 60)
print('【CANN 软件栈结构探查】')
print('=' * 60)

ascend_home = os.environ.get('ASCEND_HOME', '/usr/local/Ascend')
print(f'CANN 安装根目录: {ascend_home}')

if os.path.exists(ascend_home):
    print(f'\n📂 顶层目录结构:')
    for item in sorted(os.listdir(ascend_home)):
        full = os.path.join(ascend_home, item)
        tag = '📁' if os.path.isdir(full) else '📄'
        print(f'   {tag} {item}')

# 查看关键环境变量
print(f'\n📌 关键环境变量:')
for var in ['ASCEND_HOME', 'ASCEND_TOOLKIT_HOME']:
    val = os.environ.get(var, '[未设置]')
    print(f'   {var} = {val}')

print('\n💡 说明: CANN 软件栈已预装，无需手动安装。后续直接使用 PyTorch + torch_npu 运行推理。')

---

## 第5步：NPU 硬件信息查询（npu-smi）

> **学习目标**：使用 `npu-smi` 工具查询昇腾 NPU 的硬件信息，确认芯片型号和运行状态。

### 什么是 npu-smi？

`npu-smi` 是昇腾 NPU 的系统管理接口工具，类似于 NVIDIA GPU 上的 `nvidia-smi`。它是端侧开发中**最常用的硬件查询与监控工具**。这一步对应 CANN 协同架构中"第1层硬件"与"第2层驱动"的就绪性验证——只有驱动正确安装，`npu-smi` 才能识别到 NPU。

![NPU处理器](../../images/npu_processor.png)

In [ ]:
!npu-smi info

### 字段解读

请对照上面的输出，理解每个字段的含义：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">字段</th>
<th style="text-align: left;">含义</th>
<th style="text-align: left;">调试关注点</th>
</tr>
<tr>
<td style="text-align: left;"><strong>NPU / Chip</strong></td>
<td style="text-align: left;">NPU 编号与芯片内编号</td>
<td style="text-align: left;">确认设备编号</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Name</strong></td>
<td style="text-align: left;">芯片型号（如 910B3 = Ascend 910B3）</td>
<td style="text-align: left;">确认型号与编译目标一致</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Health</strong></td>
<td style="text-align: left;">健康状态（OK = 正常）</td>
<td style="text-align: left;">非 OK 需联系平台</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Power(W)</strong></td>
<td style="text-align: left;">实时功耗（瓦特）</td>
<td style="text-align: left;">异常高/低需关注</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Temp(C)</strong></td>
<td style="text-align: left;">芯片温度（摄氏度）</td>
<td style="text-align: left;">过热会导致降频</td>
</tr>
<tr>
<td style="text-align: left;"><strong>AICore(%)</strong></td>
<td style="text-align: left;">AI Core 利用率</td>
<td style="text-align: left;">运行任务时应上升</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Memory-Usage</strong></td>
<td style="text-align: left;">显存使用量 / 总量</td>
<td style="text-align: left;">内存泄漏需排查</td>
</tr>
<tr>
<td style="text-align: left;"><strong>HBM-Usage</strong></td>
<td style="text-align: left;">高带宽显存占用</td>
<td style="text-align: left;">大模型需关注</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Process</strong></td>
<td style="text-align: left;">占用 NPU 的进程列表</td>
<td style="text-align: left;">确认任务在 NPU 上运行</td>
</tr>
</table>

> 💡 **调试要点**：如果 AICore 利用率始终为 0 且无进程记录，说明任务并未调度到 NPU 上运行。`npu-smi` 能正常输出，说明**驱动程序已正确安装**，OS 已成功识别 NPU 硬件。

![NPU](../../images/NPU.png)

---

## 第6步：CANN 运行环境验证

> **学习目标**：检查 CANN 版本、环境变量与目录结构，验证软件栈配置正确。这一步对应 CANN 协同架构中"第3层 CANN 平台"的就绪性验证。

### 6.1 查看安装信息文件

CANN 的安装信息记录在 `ascend_toolkit_install.info` 文件中。这是获取 CANN 版本最权威的来源。

In [ ]:
import os
import subprocess

print('=' * 60)
print('【CANN 安装信息】')
print('=' * 60)

# 构建候选路径列表：环境变量 → 标准路径 → 动态搜索
info_paths = []

ascend_home = os.environ.get('ASCEND_HOME_PATH', '')
ascend_toolkit_home = os.environ.get('ASCEND_TOOLKIT_HOME', '')
if ascend_home:
    info_paths.append(os.path.join(ascend_home, 'ascend_toolkit_install.info'))
if ascend_toolkit_home:
    info_paths.append(os.path.join(ascend_toolkit_home, 'ascend_toolkit_install.info'))
    info_paths.append(os.path.join(ascend_toolkit_home, 'aarch64-linux', 'ascend_toolkit_install.info'))

info_paths.extend([
    '/usr/local/Ascend/ascend-toolkit/latest/aarch64-linux/ascend_toolkit_install.info',
    '/usr/local/Ascend/ascend-toolkit/latest/ascend_toolkit_install.info',
])

try:
    result = subprocess.run(
        ['find', '/usr/local/Ascend', '-name', 'ascend_toolkit_install.info', '-type', 'f'],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        for line in result.stdout.strip().split('\n'):
            if line and line not in info_paths:
                info_paths.append(line)
except Exception:
    pass

found = False
for p in info_paths:
    if p and os.path.exists(p):
        print(f'找到安装信息文件: {p}')
        print('-' * 40)
        with open(p) as f:
            content = f.read()
            print(content)
        found = True
        break

if not found:
    print('⚠️ 未找到安装信息文件')
    print(f'   ASCEND_HOME_PATH = {ascend_home or "未设置"}')
    print(f'   ASCEND_TOOLKIT_HOME = {ascend_toolkit_home or "未设置"}')

print('\n💡 调试要点: 确认 CANN 版本与目标设备驱动版本匹配。')

### 6.2 查看关键环境变量

CANN 环境依赖多个环境变量，通常通过 `source set_env.sh` 一次性配置：

In [ ]:
print('=' * 60)
print('【关键环境变量检查】')
print('=' * 60)

env_vars = {
    'ASCEND_HOME_PATH': 'CANN 安装路径',
    'ASCEND_TOOLKIT_HOME': '开发套件目录 (含 ATC、ACL)',
    'LD_LIBRARY_PATH': '动态链接库搜索路径 (含 libascendcl.so)',
    'PYTHONPATH': 'Python 模块搜索路径 (含 acl 模块)',
    'PATH': '可执行文件搜索路径',
}

for var, desc in env_vars.items():
    val = os.environ.get(var, '')
    print(f'\n📌 {var}')
    print(f'   说明: {desc}')
    if val:
        ascend_paths = [p for p in val.split(':') if 'Ascend' in p or 'ascend' in p]
        if ascend_paths:
            print(f'   Ascend 相关路径:')
            for p in ascend_paths[:3]:
                print(f'     {p}')
        else:
            print(f'   状态: 已设置 (无 Ascend 路径)')
    else:
        print(f'   状态: ⚠️ 未设置')

print('\n💡 通常执行 source /usr/local/Ascend/ascend-toolkit/set_env.sh 配置所有变量。')

### 6.3 查看 CANN 目录结构

了解 CANN 安装目录的整体结构，有助于定位工具和库文件。

In [ ]:
print('=' * 60)
print('【CANN 目录结构】')
print('=' * 60)

cann_base = None
candidates = [
    os.environ.get('ASCEND_HOME_PATH', ''),
    os.environ.get('ASCEND_TOOLKIT_HOME', ''),
    '/usr/local/Ascend/ascend-toolkit/latest',
    os.path.join(os.environ.get('HOME', ''), 'Ascend/ascend-toolkit/latest'),
]

for c in candidates:
    if c and os.path.exists(c):
        cann_base = c
        break

if cann_base:
    print(f'CANN 安装目录: {cann_base}')
    print('-' * 40)
    try:
        for item in sorted(os.listdir(cann_base)):
            item_path = os.path.join(cann_base, item)
            tag = '📁' if os.path.isdir(item_path) else '📄'
            print(f'  {tag} {item}')
    except PermissionError:
        print('  ⚠️ 无权限读取目录内容')
else:
    print(f'⚠️ 未找到 CANN 安装目录')

# 查找 ATC 工具
print('\n📌 查找 ATC 模型转换工具:')
atc_found = False
try:
    result = subprocess.run(['which', 'atc'], capture_output=True, text=True, timeout=3)
    if result.returncode == 0 and result.stdout.strip():
        print(f'  ✅ 通过 which 找到: {result.stdout.strip()}')
        atc_found = True
except Exception:
    pass
if not atc_found and cann_base:
    for root, dirs, files in os.walk(cann_base):
        if 'atc' in files and 'bin' in root:
            print(f'  ✅ 找到: {os.path.join(root, "atc")}')
            atc_found = True
            break
if not atc_found:
    print('  ⚠️ 未找到 ATC，可能仅安装了运行时(nnrt)')

---

## 第7步：ACL 编程与 HelloWorld 示例

> **学习目标**：通过 HelloWorld 示例理解 ACL（AscendCL）编程基本流程，验证 CANN 软件栈端到端通路是否正常。

### 7.1 HelloWorld 示例的意义

HelloWorld 示例验证了 **CANN 软件栈、编译器、驱动与 NPU 硬件** 之间的完整通路，属于最底层的算子级验证。

```
你的代码 → AscendCL 编译器 → NPU 驱动 → AI Core 硬件 → 输出结果
```

如果 HelloWorld 能成功运行并输出，说明从软件到硬件的整条链路都是通的——这正是第1步讲解的"固件→驱动→CANN"协同架构在端到端层面的最终验证。

### 7.2 ACL 编程基本流程

ACL（Ascend Computing Language）编程的标准流程为 8 步：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">步骤</th>
<th style="text-align: left;">API</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>aclInit()</code></td>
<td style="text-align: left;">初始化 ACL 运行时</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><code>aclrtSetDevice()</code></td>
<td style="text-align: left;">指定计算设备</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;"><code>aclrtCreateStream()</code></td>
<td style="text-align: left;">创建执行流</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">执行计算任务</td>
<td style="text-align: left;">下发算子到 NPU</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;"><code>aclrtSynchronizeStream()</code></td>
<td style="text-align: left;">同步等待完成</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;"><code>aclrtDestroyStream()</code></td>
<td style="text-align: left;">销毁执行流</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;"><code>aclrtResetDevice()</code></td>
<td style="text-align: left;">释放设备</td>
</tr>
<tr>
<td style="text-align: left;">8</td>
<td style="text-align: left;"><code>aclFinalize()</code></td>
<td style="text-align: left;">去初始化</td>
</tr>
</table>

上表展示了 ACL 编程的标准 8 步流程。这是所有昇腾 NPU 应用的基本骨架：先初始化运行时（步骤 1），设置计算设备（步骤 2），创建执行流（步骤 3），下发计算任务到 NPU（步骤 4），同步等待完成（步骤 5），最后依次销毁流、释放设备、去初始化（步骤 6-8）。这一流程类似于 CUDA 编程模型，确保资源正确分配和释放。

### 7.3 示例代码解析

**算子核函数（hello_world.cpp）**：

```cpp
#include "kernel_operator.h"

extern "C" __global__ __aicore__ void hello_world()
{
    AscendC::printf("Hello World!!!\n");
}

void hello_world_do(uint32_t blockDim, void *stream)
{
    hello_world<<<blockDim, nullptr, stream>>>();
}
```

**主函数（main.cpp）**：

```cpp
#include "acl/acl.h"
extern void hello_world_do(uint32_t coreDim, void *stream);

int32_t main(int32_t argc, char const *argv[])
{
    aclInit(nullptr);                    // 1. 初始化 ACL
    int32_t deviceId = 0;
    aclrtSetDevice(deviceId);            // 2. 指定计算设备
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);          // 3. 创建 Stream
    
    constexpr uint32_t blockDim = 8;     // 8 个 AI Core 并行
    hello_world_do(blockDim, stream);    // 4. 执行计算任务
    aclrtSynchronizeStream(stream);      // 5. 同步等待

    aclrtDestroyStream(stream);          // 6. 销毁 Stream
    aclrtResetDevice(deviceId);          // 7. 释放设备
    aclFinalize();                       // 8. 去初始化
    return 0;
}
```

### 7.4 操作步骤（在 Terminal 中执行）

```bash
# 1. 下载官方示例仓库
git clone https://gitee.com/ascend/samples.git

# 2. 进入 HelloWorld 示例目录
cd ~/samples/operator/ascendc/0_introduction/0_helloworld

# 3. 修改编译目标芯片型号
vim CMakeLists.txt
# 将 SOC_VERSION 默认值修改为 Ascend910B3（与沙箱芯片一致）

# 4. 编译并运行
sudo ./run.sh -v Ascend910B3
```

> ⚠️ **关键调试点**：必须将 CMakeLists.txt 中的编译目标从默认的 Ascend910P3 改为与沙箱一致的芯片型号（如 Ascend910B3），否则编译会报芯片型号不匹配错误。

### 7.5 预期输出

```text
CANN Version: 9.0.0
AIV-0: Hello World!!!
AIV-1: Hello World!!!
AIV-2: Hello World!!!
AIV-3: Hello World!!!
AIV-4: Hello World!!!
AIV-5: Hello World!!!
AIV-6: Hello World!!!
AIV-7: Hello World!!!
```

8 个 AIV 核（block dim=8）分别打印 Hello World!!!，说明算子已成功在 NPU 上加载并执行，**沙箱环境完全可用**。

### 7.6 在 Notebook 中验证 ACL 环境

虽然 HelloWorld 需要 C++ 编译，但我们可以在 Notebook 中用 Python 验证 ACL 环境，并模拟 HelloWorld 的多核并行输出：

In [ ]:
import os
import subprocess

print('=' * 60)
print('【ACL 环境验证 + HelloWorld 模拟】')
print('=' * 60)

# 1. 验证 ACL 模块
print('步骤1: 验证 ACL 模块')
try:
    import acl
    print('  ✅ acl 模块导入成功')
    version = acl.get_version()
    print(f'  ACL 版本标识: {version}')
    version_map = {1: 'CANN 6.x', 2: 'CANN 7.x', 3: 'CANN 8.x', 4: 'CANN 9.x'}
    if isinstance(version, tuple) and len(version) >= 1:
        print(f'  对应系列: {version_map.get(version[0], "未知")}')
except ImportError as e:
    print(f'  ⚠️ acl 模块导入失败: {e}')

# 2. 读取 CANN 版本
print('\n步骤2: 读取 CANN 版本')
cann_ver = '未知'
_info_paths = []
_ah = os.environ.get('ASCEND_HOME_PATH', '')
_ath = os.environ.get('ASCEND_TOOLKIT_HOME', '')
if _ah:
    _info_paths.append(os.path.join(_ah, 'ascend_toolkit_install.info'))
if _ath:
    _info_paths.append(os.path.join(_ath, 'ascend_toolkit_install.info'))
    _info_paths.append(os.path.join(_ath, 'aarch64-linux', 'ascend_toolkit_install.info'))
_info_paths.extend([
    '/usr/local/Ascend/ascend-toolkit/latest/aarch64-linux/ascend_toolkit_install.info',
    '/usr/local/Ascend/ascend-toolkit/latest/ascend_toolkit_install.info',
])
try:
    _r = subprocess.run(['find', '/usr/local/Ascend', '-name', 'ascend_toolkit_install.info', '-type', 'f'],
                        capture_output=True, text=True, timeout=10)
    if _r.returncode == 0:
        for _l in _r.stdout.strip().split('\n'):
            if _l and _l not in _info_paths:
                _info_paths.append(_l)
except Exception:
    pass
for p in _info_paths:
    if p and os.path.exists(p):
        with open(p) as f:
            for line in f:
                if line.startswith('version='):
                    cann_ver = line.split('=', 1)[1].strip()
        break
print(f'  CANN 版本: {cann_ver}')

# 3. 模拟 HelloWorld 输出（8 个 AIV 核并行）
print('\n步骤3: 模拟 HelloWorld 多核并行输出')
print(f'CANN Version: {cann_ver}')
block_dim = 8  # 与 main.cpp 中的 blockDim = 8 一致
for i in range(block_dim):
    print(f'AIV-{i}: Hello World!!!')

print('\n✅ 8 个 AIV 核正常输出，说明 NPU 硬件与 CANN 软件栈端到端通路正常。')
print('   这验证了第1步讲解的"固件→驱动→CANN"协同架构在端到端层面完全打通。')



## 第8步：昇腾香橙派查询命令与结果

> **学习目标**：在昇腾香橙派 AIPro 开发板上通过命令查询 NPU 硬件状态与 CANN 软件版本信息，验证“固件→驱动→CANN”链路的一致性。

### 8.1 香橙派 AIPro 开发板简介

OrangePi AIPro 开发板由香橙派与华为联合打造，搭载昇腾 AI 处理器与 4 核 64 位 ARM 处理器，提供 8 TOPS INT8 算力，广泛应用于教育、机器人、无人机及边缘 AI 计算等场景。

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">核心配置</th>
<th style="text-align: left;">规格</th>
</tr>
<tr>
<td style="text-align: left;">AI 算力</td>
<td style="text-align: left;">8 TOPS INT8</td>
</tr>
<tr>
<td style="text-align: left;">FP16 算力</td>
<td style="text-align: left;">4 TFLOPS</td>
</tr>
<tr>
<td style="text-align: left;">CPU</td>
<td style="text-align: left;">4 核 64 位 ARM</td>
</tr>
<tr>
<td style="text-align: left;">内存</td>
<td style="text-align: left;">8/16 GB LPDDR4X</td>
</tr>
<tr>
<td style="text-align: left;">操作系统</td>
<td style="text-align: left;">Ubuntu 22.04 / openEuler 22.03</td>
</tr>
<tr>
<td style="text-align: left;">CANN 版本</td>
<td style="text-align: left;">8.0.0</td>
</tr>
</table>

![昇腾产品](images/Ascend%20Products.png)

### 8.2 查询命令一：npu-smi info

**命令**：

```bash
npu-smi info
```

**说明**：`npu-smi` 是昇腾 NPU 系统管理接口工具，用于查询 NPU 设备的芯片型号、健康状态、功耗、温度、AI Core 利用率等硬件信息。在香橙派上执行该命令可确认驱动已正确安装且 NPU 设备就绪。

**执行结果**：

![npu-smi info 查询结果](images/orangepi_npu_smi.png)

从结果可以看到 NPU 设备被系统正确识别，芯片健康状态为 OK，说明驱动层工作正常。

### 8.3 查询命令二：查看 CANN 安装信息

**命令**：

```bash
cat /usr/local/Ascend/ascend-toolkit/latest/aarch64-linux/ascend_toolkit_install.info
```

**说明**：该命令读取 CANN Toolkit 的安装信息文件，获取软件包版本、安装路径、适用架构等关键信息，用于确认当前开发板上的 CANN 版本。

**执行结果**：

![CANN 安装信息查询结果](images/orangepi_cann_install_info.png)

从结果可以确认 CANN Toolkit 的版本号与安装路径，验证软件层安装正确。

### 8.4 查询过程小结

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">步骤</th>
<th style="text-align: left;">命令</th>
<th style="text-align: left;">查询层级</th>
<th style="text-align: left;">验证内容</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;"><code>npu-smi info</code></td>
<td style="text-align: left;">硬件层</td>
<td style="text-align: left;">NPU 芯片型号、健康状态、驱动是否就绪</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;"><code>cat .../ascend_toolkit_install.info</code></td>
<td style="text-align: left;">软件层</td>
<td style="text-align: left;">CANN Toolkit 版本号、安装路径</td>
</tr>
</table>

1. 首先通过 `npu-smi info` 确认 NPU 硬件设备被系统正确识别，芯片健康状态为 OK。
2. 然后通过查看 `ascend_toolkit_install.info` 文件确认 CANN Toolkit 的版本号与安装路径。
3. 两步查询分别覆盖“硬件层”与“软件层”，验证了香橙派上**固件→驱动→CANN** 链路的一致性。

> **结论**：通过 `npu-smi info` 与 `ascend_toolkit_install.info` 两条命令，分别从硬件层和软件层确认了香橙派上 NPU 设备就绪、CANN 版本正确，整条“固件→驱动→CANN”链路完全打通。

![训练与推理](../../images/training_inference.png)

---

## 第9步：总结与思考

### 本实验完成的工作

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">步骤</th>
<th style="text-align: left;">内容</th>
<th style="text-align: left;">关键收获</th>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">三者关系讲解</td>
<td style="text-align: left;">理解 CANN、OS 与驱动自下而上的协同架构</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">CANN 核心模块</td>
<td style="text-align: left;">掌握 CANN 四层软件栈及各层核心模块功能</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">环境快速确认</td>
<td style="text-align: left;">确认 CANN + torch_npu + NPU 就绪</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">CANN 软件栈探查</td>
<td style="text-align: left;">探查 CANN 安装目录与关键环境变量</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">NPU 硬件信息查询</td>
<td style="text-align: left;">使用 npu-smi 查询 NPU 状态，验证驱动就绪</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">CANN 运行环境验证</td>
<td style="text-align: left;">检查 CANN 版本、环境变量、目录结构</td>
</tr>
<tr>
<td style="text-align: left;">7</td>
<td style="text-align: left;">ACL 编程与 HelloWorld</td>
<td style="text-align: left;">验证 CANN 软件栈端到端通路</td>
</tr>
<tr>
<td style="text-align: left;">8</td>
<td style="text-align: left;">香橙派查询验证</td>
<td style="text-align: left;">在香橙派上查询 NPU 与 CANN 版本信息（见 <code>新建 Microsoft Word 文档.docx</code>）</td>
</tr>
</table>

### 关键实验结果

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">检查项</th>
<th style="text-align: left;">结果</th>
</tr>
<tr>
<td style="text-align: left;"><strong>芯片型号</strong></td>
<td style="text-align: left;">Ascend 910B3</td>
</tr>
<tr>
<td style="text-align: left;"><strong>健康状态</strong></td>
<td style="text-align: left;">OK</td>
</tr>
<tr>
<td style="text-align: left;"><strong>CANN 版本</strong></td>
<td style="text-align: left;">9.0.0</td>
</tr>
<tr>
<td style="text-align: left;"><strong>Python 版本</strong></td>
<td style="text-align: left;">3.11</td>
</tr>
<tr>
<td style="text-align: left;"><strong>PyTorch + torch_npu</strong></td>
<td style="text-align: left;">可用</td>
</tr>
<tr>
<td style="text-align: left;"><strong>HelloWorld 验证</strong></td>
<td style="text-align: left;">8 个 AIV 核正常输出</td>
</tr>
<tr>
<td style="text-align: left;"><strong>端到端通路</strong></td>
<td style="text-align: left;">固件→驱动→CANN 完全打通</td>
</tr>
</table>

### 课后思考

1. **为什么部署昇腾环境必须遵循"固件→驱动→CANN"的顺序？如果顺序颠倒会发生什么？**

   > 提示：CANN 安装时会检测驱动是否存在，若先装 CANN 再装驱动，CANN 找不到 NPU 设备会安装失败或配置不完整。

2. **CANN 的四层软件栈中，哪一层是开发者最常接触的？为什么？**

   > 提示：应用开发与编程接口层（AscendCL、GE、ATC），因为开发者直接使用这些 API 和工具进行应用开发和模型部署。

3. **AscendCL 与 ATC 各自承担什么角色？分别适合什么场景？**

   > 提示：AscendCL 提供运行时编程接口（推理/计算），ATC 提供模型转换工具（部署前编译）。前者是运行时，后者是编译时。

4. **HAL（硬件抽象层）在 CANN 软件栈中起什么作用？**

   > 提示：屏蔽底层硬件差异，使得同一套 AscendCL 代码可以在不同型号昇腾芯片（如 310B4、910B3）上运行。

> 💡 思考题参考答案见 `answer/` 目录。

### 建立的昇腾软件开发思路

```
1. 理解架构 → 固件→驱动→CANN→框架 自下而上层层依赖
2. 确认硬件 → npu-smi info (芯片型号、健康状态)
3. 确认软件 → 检查 CANN 版本、环境变量
4. 验证通路 → 运行 HelloWorld 示例
5. 版本管理 → 严格匹配固件/驱动/CANN 版本
6. 香橙派查询 → 查询 NPU 与 CANN 版本，验证链路一致
```

![AI开发](../../images/artificial_intelligence_development.png)

---

## 参考资料

- [昇腾社区 - CANN 文档](https://hiascend.com/document)
- [昇腾社区 - CANN 下载页面](https://www.hiascend.com/developer/download/community/result?module=cann)
- [PyTorch 昇腾 NPU 适配指南](https://hiascend.com/document/detail/zh/Pytorch/Pytorch)
- [昇腾官方 Samples 仓库](https://gitee.com/ascend/samples)
- [GitCode 代码托管平台](https://gitcode.com/)
- [香橙派 AIpro 官方文档](http://www.orangepi.cn/)
- [CANN 软件安装指南](https://hiascend.com/document/redirect/CannCommunityInstSoftware)
- 课件：《实验3.3 昇腾CANN基础操作实验》